# Lesson 38: CrewAI II — memory, embeddings, knowledge, planning, reasoning, Flows and the CLI 🧠🌊
> 🎓 **Advanced track** — best for Years 11–13. Students should be comfortable with Python functions, lists and dictionaries.

**Learning goals**
1. Choose an **embedder** and give crews **memory** that lasts between runs
2. Give agents **knowledge** from text and files (CrewAI's built-in RAG)
3. Turn on **planning** (crew-level) and **reasoning** (agent-level)
4. Build a **Flow**: event-driven steps with state, routing and crews inside
5. Use the **CrewAI CLI** to create a real project

**⏱️ Timing:** embeddings & memory 15' | knowledge 10' | planning & reasoning 10' | Flows 15' | CLI 10'

> ⚙️ **GPU required:** Runtime → Change runtime type → **T4 GPU**. Multi-agent crews make many LLM calls, so this lesson uses `qwen2.5:7b` (about 5 GB to download). For a faster but less reliable run, change it to `qwen2.5:3b`.

In [ ]:
# 🦙 Start a free LLM server *inside* Colab (Ollama). No API key or account needed.
# ⏳ First run: about 2–3 minutes (installs Ollama and downloads the model).
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1
import subprocess, time
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

MODEL = "qwen2.5:7b"     # smarter at using tools. "qwen2.5:3b" downloads faster but makes more mistakes
!ollama pull {MODEL} > /dev/null 2>&1
print("✅ Ollama is running with", MODEL)

In [ ]:
!ollama pull nomic-embed-text > /dev/null 2>&1      # a small embedding model
!pip -q install crewai
import os, warnings; warnings.filterwarnings("ignore")
os.environ["CREWAI_TRACING_ENABLED"] = "false"     # don't send traces anywhere / don't ask about it
os.environ["OTEL_SDK_DISABLED"] = "true"
from crewai import Agent, Task, Crew, Process, LLM

llm = LLM(model=f"ollama/{MODEL}", base_url="http://localhost:11434", temperature=0.2)
print("✅ CrewAI ready, using", MODEL)

## Part 1: Embeddings in CrewAI 🧭
Memory and knowledge both work by **embedding** text and searching by similarity (Lessons 22 and 31). CrewAI lets you pick the **embedder provider**: `openai`, `ollama`, `huggingface`, `sentence-transformer`, `google-generativeai`, `cohere`, `voyageai` and more.

> ⚠️ If you don't choose, CrewAI defaults to OpenAI — which needs a paid key. We'll use a free local embedder through Ollama.

In [ ]:
EMBEDDER = {"provider": "ollama", "config": {"model_name": "nomic-embed-text", "url": "http://localhost:11434/api/embeddings"}}

# quick check: embed two sentences through Ollama and compare
import requests, numpy as np
def embed(text):
    return np.array(requests.post("http://localhost:11434/api/embeddings", json={"model": "nomic-embed-text", "prompt": text}).json()["embedding"])
a, b, c = embed("The cat sat on the mat."), embed("A kitten rested on the rug."), embed("Stock prices fell today.")
cos = lambda u, v: float(u @ v / np.linalg.norm(u) / np.linalg.norm(v))
print(f"cat/kitten: {cos(a, b):.2f} | cat/stocks: {cos(a, c):.2f} | vector size: {len(a)}")

## Part 2: Memory 💾
CrewAI's **memory** stores what crews learn and **recalls** relevant memories in later runs. Each memory gets a **scope** (topic folder), an **importance** score and a timestamp; recall mixes **similarity**, **recency** and **importance**.

> Like a student's notebook: you write down important things, and later look up what's relevant (and recent).

In [ ]:
from crewai.memory.unified_memory import Memory
os.environ["CREWAI_STORAGE_DIR"] = "./crew_storage"            # where memories are saved on disk

memory = Memory(llm=llm, embedder=EMBEDDER)                      # LLM + embedder are both local
memory.remember("Ms Patel's Year 10 class prefers short, visual explanations.", importance=0.9)
memory.remember("The class finished the unit on chemical reactions last week.")
memory.remember("The school's science fair is on 14 August.")

for match in memory.recall("How should I explain things to Year 10?", limit=2, depth="shallow"):
    print(f"🔎 {match.record.content}   (scope {match.record.scope})")

In [ ]:
tutor = Agent(role="Lesson Planner", goal="Plan lessons that suit this particular class",
              backstory="You plan science lessons and always use what you remember about the class.", llm=llm)
plan = Task(description="Plan a 3-step starter activity for Ms Patel's Year 10 class on the topic: {topic}.",
            expected_output="A 3-step starter activity", agent=tutor)

crew = Crew(agents=[tutor], tasks=[plan], memory=memory)          # this crew can read AND add memories
print(crew.kickoff(inputs={"topic": "acids and bases"}).raw)

### 🧩 Exercise 1
1. Run the crew a second time with a new topic. Did it use what it remembered about the class?
2. `memory.recall("science fair")` — does it find the date?
3. Use `memory.forget(...)` or `memory.reset()` to clear memories. Why should users be able to see and delete what an AI remembers about them?

## Part 3: Knowledge 📚
**Knowledge** gives agents reference material — built-in RAG. Sources can be strings, text files, PDFs, CSV, Excel, JSON… Files go in a folder called **`knowledge/`**.

In [ ]:
from crewai.knowledge.source.string_knowledge_source import StringKnowledgeSource
from crewai.knowledge.source.text_file_knowledge_source import TextFileKnowledgeSource

os.makedirs("knowledge", exist_ok=True)
open("knowledge/lab_rules.txt", "w").write("""Lab rules for Years 9-10:
1. Safety glasses must be worn whenever chemicals or heat are used.
2. Long hair must be tied back near Bunsen burners.
3. No food or drink in the lab.
4. Report every breakage or spill to the teacher immediately.
5. Hydrochloric acid stronger than 1 mol/L may only be handled by the teacher.""")

school_facts = StringKnowledgeSource(content="The science department has 4 labs (S1–S4). Lab S3 has the fume cupboard. "
                                             "Practical lessons are 100 minutes long.")
lab_rules = TextFileKnowledgeSource(file_paths=["lab_rules.txt"])          # path relative to the knowledge/ folder

lab_assistant = Agent(role="Lab Assistant", goal="Answer questions about our school's science labs accurately",
                      backstory="You only answer from the school's lab rules and facts.",
                      knowledge_sources=[school_facts, lab_rules], embedder=EMBEDDER, llm=llm)
q = Task(description="A Year 10 student asks: '{question}'", expected_output="A short, accurate answer", agent=lab_assistant)
lab_crew = Crew(agents=[lab_assistant], tasks=[q], embedder=EMBEDDER)
for question in ["Which lab should we use for an experiment that makes fumes?",
                 "Can I handle 2 mol/L hydrochloric acid myself?"]:
    print(f"❓ {question}\n🤖 {lab_crew.kickoff(inputs={'question': question}).raw}\n")

**Memory vs knowledge:** **knowledge** is reference material you give the agent up front (like a textbook). **Memory** is what the crew learns and stores while working (like your own notes).

## Part 4: Planning and reasoning 🧩
- **Planning** (`Crew(planning=True)`): before starting, a planner writes a step-by-step plan for **every task** and adds it to the task descriptions
- **Reasoning** (`Agent(reasoning=True)`): before a task, the agent **reflects** and writes its own plan, checking whether it's ready

Both cost extra LLM calls (and time), but help with complicated tasks.

In [ ]:
researcher = Agent(role="Trip Researcher", goal="Plan safe, educational school trips",
                   backstory="You organise school trips and think about cost, safety and learning.",
                   reasoning=True, max_reasoning_attempts=1, llm=llm, verbose=True)
trip = Task(description="Plan a one-day Year 9 science trip to a botanical garden for 60 students and 4 teachers. "
                        "Include a timetable, 2 learning activities, and a cost estimate assuming $12 entry per student.",
            expected_output="Timetable, activities, cost estimate", agent=researcher)
planned_crew = Crew(agents=[researcher], tasks=[trip], planning=True, planning_llm=llm)
print(planned_crew.kickoff().raw)

### 🧩 Exercise 2
Run the same task with `planning=False` and `reasoning=False`. Compare quality and time. When is the extra thinking worth it?

## Part 5: Flows — event-driven workflows 🌊
A **Crew** is a team of agents with some freedom. A **Flow** is a precise **workflow** you control with Python: steps, shared **state**, branches — and you can run whole crews inside steps. (Similar idea to LangGraph in Lesson 34.)
| Decorator | Meaning |
|---|---|
| `@start()` | The first step(s) |
| `@listen(step)` | Run when `step` finishes |
| `@router(step)` | Choose which branch runs next by returning a label |

In [ ]:
from crewai.flow.flow import Flow, start, listen, router
from pydantic import BaseModel

class EssayState(BaseModel):
    topic: str = ""
    essay: str = ""
    score: int = 0
    feedback: str = ""

class EssayFlow(Flow[EssayState]):
    @start()
    def write_essay(self):
        writer = Agent(role="Student Writer", goal="Write a short persuasive paragraph", backstory="A Year 10 student.", llm=llm)
        task = Task(description=f"Write a 5-sentence persuasive paragraph: {self.state.topic}", expected_output="A paragraph", agent=writer)
        self.state.essay = Crew(agents=[writer], tasks=[task]).kickoff().raw     # a crew inside a flow step
        print("✍️ essay written")

    @listen(write_essay)
    def mark_essay(self):
        reply = llm.call(f"Score this paragraph from 1 to 10 for persuasiveness. Reply with the number only.\n\n{self.state.essay}")
        digits = "".join(ch for ch in reply if ch.isdigit())
        self.state.score = int(digits[:2]) if digits else 0
        print("🧑‍🏫 score:", self.state.score)

    @router(mark_essay)
    def decide(self):
        return "excellent" if self.state.score >= 8 else "needs_work"

    @listen("excellent")
    def celebrate(self):
        self.state.feedback = "🎉 Great work — submitted to the school newsletter!"

    @listen("needs_work")
    def give_feedback(self):
        self.state.feedback = llm.call(f"Give 2 specific tips to make this more persuasive:\n\n{self.state.essay}")

flow = EssayFlow()
flow.kickoff(inputs={"topic": "Schools should have a 4-day week"})
print("\n📄 ESSAY:\n", flow.state.essay, "\n\n💬 FEEDBACK:\n", flow.state.feedback)

In [ ]:
# Draw the flow (saves an interactive HTML file you can download from the Files panel)
try:
    flow.plot("essay_flow")
    print("Saved essay_flow.html — open it from the Files panel on the left")
except Exception as e:
    print("Plotting not available in this version:", e)

### 🧩 Exercise 3
Add a loop: if the score is below 8, rewrite the essay using the feedback and mark it again (at most 2 rewrites). Keep a `rewrites` counter in the state.

## Part 6: The CrewAI CLI ⌨️
Real projects aren't single notebooks. The **CLI** (command-line interface) creates a project folder with agents and tasks in **YAML** files, separate from the Python code.

In [ ]:
!crewai --help | head -30

In [ ]:
!crewai create flow study_helper > /dev/null 2>&1
!find study_helper -not -path "*/.git*" -type f | sort
print("\n--- agents.yaml (roles live in config, not code) ---")
!cat study_helper/src/study_helper/crews/*/config/agents.yaml

Useful commands (run them inside the project folder):
| Command | What it does |
|---|---|
| `crewai create crew <name>` | New crew project (asks which LLM provider to use) |
| `crewai create flow <name>` | New flow project |
| `crewai install` | Install the project's dependencies |
| `crewai run` | Run the crew or flow |
| `crewai test` | Run the crew several times and score the results |
| `crewai reset-memories --all` | Clear stored memories and knowledge |
| `crewai log-tasks-outputs` / `crewai replay -t <task_id>` | See past task outputs and re-run from a task |

To use the local model in a CLI project, put these in the project's `.env` file:
```
MODEL=ollama/qwen2.5:7b
API_BASE=http://localhost:11434
```

### 🚀 Challenge
Open `study_helper/src/study_helper/crews/*/config/tasks.yaml`, change the tasks into a "make revision notes on {topic}" task, then `cd study_helper && crewai install && crewai run`.

## 📌 Summary
- Pick an **embedder** (default is OpenAI — set a local one to avoid needing a key)
- **Memory** = notes the crew writes and recalls (similarity + recency + importance)
- **Knowledge** = reference material (strings, files) the agents search
- **Planning** (crew) and **reasoning** (agent) add thinking before acting
- **Flows** = precise, stateful workflows with `@start`, `@listen`, `@router`; crews can run inside
- The **CLI** scaffolds real projects with YAML config